# DMET-SSTKG — Phase-Initialized S-STKG Motif Extractor

This notebook reproduces the phase-initialized motif-extraction stage used by
DMET-SSTKG. It must be run after Notebook 08.

The user edits only three paths in the configuration cell:

- `BASE_PATH`: the existing 20-case source S-STKG directory;
- `PHASE_RESULTS_DIR`: the selected completed run directory from Notebook 08;
- `RESULTS_ROOT`: a writable directory for motif-extractor outputs.

No cache path is entered by the user. The notebook automatically uses the
source binary-cache location created by Notebook 08:

```text
<PHASE_RESULTS_DIR parent>/source_phase_classifier_binary_cache/
```

When that cache is missing, incomplete, or incompatible, it is rebuilt
automatically from `BASE_PATH`. Before training, the notebook verifies that the
current cache ordering exactly matches Notebook 08's
`fragment_metadata.csv` and `data_split.npz`.

The retained configuration uses:

- the same case-disjoint 14/3/3 split produced by Notebook 08;
- a frozen first GATv2 layer;
- trainable upper graph-temporal layers and motif components;
- motif window length 16 and stride 8;
- 128-dimensional normalized motif signatures;
- inverse motif-window-frequency class weights;
- label smoothing 0.05;
- `L_motif + 0.02 L_sig + 0.20 L_seq`;
- AdamW with motif/head learning rate `5e-4` and backbone learning rate `1e-4`;
- checkpoint selection by validation Macro F1, with validation accuracy as the
  tie-breaker.

The final checkpoint is saved as:

```text
best_phase_initialized_motif_extractor.pt
```


## Environment

Use the same Python environment as Notebook 08. The main dependencies are:

```bash
python -m pip install torch==2.4.1
python -m pip install torch-geometric pandas numpy matplotlib scikit-learn tqdm jupyter
```

Absolute paths are recommended so execution does not depend on the directory
from which Jupyter was launched.


In [ ]:
from __future__ import annotations

import hashlib
import json
import random
import shutil
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.cluster import KMeans
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    silhouette_score,
)
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATv2Conv
from tqdm.auto import tqdm


In [ ]:
# =========================================================
# 0) CONFIGURATION — EDIT ONLY THESE THREE PATHS
# =========================================================

# Existing source S-STKG directory containing Case_01, ..., Case_20.
BASE_PATH = Path(
    "/path/to/source_sstkg_directory"
)

# Selected completed output directory produced by Notebook 08.
# It must contain:
#   best_sstkg_phase_classifier.pt
#   data_split.npz
#   fragment_metadata.csv
PHASE_RESULTS_DIR = Path(
    "/path/to/selected_phase_classifier_run"
)

# Writable directory for motif checkpoints, reports, and figures.
RESULTS_ROOT = Path(
    "/path/to/motif_output_directory"
)

# No prebuilt cache path is entered by the user. Notebook 08 creates this
# cache beside its timestamped run directories. If it is unavailable or
# incomplete, this notebook reconstructs it automatically from BASE_PATH.
BIN_PATH = (
    PHASE_RESULTS_DIR.parent
    / "source_phase_classifier_binary_cache"
)

PHASE_CKPT = (
    PHASE_RESULTS_DIR
    / "best_sstkg_phase_classifier.pt"
)
SPLIT_NPZ = (
    PHASE_RESULTS_DIR
    / "data_split.npz"
)
PHASE_METADATA_CSV = (
    PHASE_RESULTS_DIR
    / "fragment_metadata.csv"
)

SEED = 42

CHUNK_SIZE = 100
MIN_CHUNK_FRAMES = 10
NUM_PHASES = 8
HIDDEN_DIM = 128
MOTIF_DIM = 128
NUM_EPOCHS = 30

AUX_WEIGHT = 0.02
SEQ_WEIGHT = 0.20
LABEL_SMOOTHING = 0.05

MOTIF_LR = 5e-4
BACKBONE_LR = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

REMOVE_CASE_ID_FEATURE = True
REMOVE_G_RATIO_FEATURE = False
REMOVE_P_RATIO_FEATURE = False

CASE_ID_COL = 7
G_RATIO_COL = 8
P_RATIO_COL = 9
VIRTUAL_NODE_COL = 10

EXPECTED_NUM_CASES = 20
TRAIN_CASES_N = 14
VAL_CASES_N = 3
TEST_CASES_N = 3

CACHE_SCHEMA_VERSION = 2
FRAME_KEY_TOLERANCE = 1e-6

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PHASE_NAMES = [
    "TrocarPlacement",
    "Preparation",
    "CalotTriangleDissection",
    "ClippingCutting",
    "GallbladderDissection",
    "GallbladderPackaging",
    "CleaningCoagulation",
    "GallbladderRetraction",
]


def paths_overlap(path_a, path_b):
    """True when paths are equal or one is an ancestor of the other."""
    path_a = Path(path_a).expanduser().resolve()
    path_b = Path(path_b).expanduser().resolve()

    return (
        path_a == path_b
        or path_a in path_b.parents
        or path_b in path_a.parents
    )


def discover_case_directories(root_dir):
    root_dir = Path(root_dir).expanduser().resolve()

    if not root_dir.is_dir():
        return []

    return sorted(
        path
        for path in root_dir.iterdir()
        if (
            path.is_dir()
            and path.name.startswith("Case_")
        )
    )


def validate_input_paths():
    source_root = Path(
        BASE_PATH
    ).expanduser().resolve()
    phase_run = Path(
        PHASE_RESULTS_DIR
    ).expanduser().resolve()
    cache_root = Path(
        BIN_PATH
    ).expanduser().resolve()
    results_root = Path(
        RESULTS_ROOT
    ).expanduser().resolve()

    if not source_root.is_dir():
        raise FileNotFoundError(
            "Source S-STKG directory was not found:\n"
            f"{source_root}\n\n"
            "Update BASE_PATH in the configuration cell."
        )

    case_directories = discover_case_directories(
        source_root
    )
    if len(case_directories) != EXPECTED_NUM_CASES:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_NUM_CASES} source Case_* "
            f"folders but found {len(case_directories)} under:\n"
            f"{source_root}\n\n"
            "Detected cases: "
            f"{[path.name for path in case_directories]}"
        )

    if not phase_run.is_dir():
        raise FileNotFoundError(
            "The selected Notebook 08 result directory was not found:\n"
            f"{phase_run}\n\n"
            "Update PHASE_RESULTS_DIR in the configuration cell."
        )

    required_phase_artifacts = [
        Path(PHASE_CKPT).expanduser().resolve(),
        Path(SPLIT_NPZ).expanduser().resolve(),
        Path(PHASE_METADATA_CSV).expanduser().resolve(),
    ]
    missing_artifacts = [
        path
        for path in required_phase_artifacts
        if not path.is_file()
    ]
    if missing_artifacts:
        raise FileNotFoundError(
            "The selected Notebook 08 run is incomplete. Missing:\n"
            + "\n".join(
                str(path)
                for path in missing_artifacts
            )
        )

    if paths_overlap(source_root, cache_root):
        raise RuntimeError(
            "The automatically managed binary cache must be separate "
            "from BASE_PATH.\n"
            f"BASE_PATH: {source_root}\n"
            f"BIN_PATH : {cache_root}"
        )

    if paths_overlap(source_root, results_root):
        raise RuntimeError(
            "RESULTS_ROOT must be separate from BASE_PATH and must not "
            "contain the source data.\n"
            f"BASE_PATH   : {source_root}\n"
            f"RESULTS_ROOT: {results_root}"
        )

    if paths_overlap(source_root, phase_run):
        raise RuntimeError(
            "PHASE_RESULTS_DIR must be separate from BASE_PATH.\n"
            f"BASE_PATH        : {source_root}\n"
            f"PHASE_RESULTS_DIR: {phase_run}"
        )

    if (
        TRAIN_CASES_N
        + VAL_CASES_N
        + TEST_CASES_N
        != EXPECTED_NUM_CASES
    ):
        raise RuntimeError(
            "Train/validation/test case counts do not sum to "
            f"{EXPECTED_NUM_CASES}."
        )

    results_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"[OK] Source S-STKG       : {source_root}")
    print(f"[OK] Source cases        : {len(case_directories)}")
    print(f"[OK] Phase run           : {phase_run}")
    print(f"[OK] Managed binary cache: {cache_root}")
    print(f"[OK] Motif results root  : {results_root}")
    print(f"[OK] Device              : {DEVICE}")

    return case_directories


SOURCE_CASE_DIRECTORIES = validate_input_paths()


In [ ]:
# =========================================================
# 1) REPRODUCIBILITY AND LABEL HELPERS
# =========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def safe_label_to_zero_based(raw_label, num_classes=8):
    raw_label = int(raw_label)

    if 1 <= raw_label <= num_classes:
        return raw_label - 1

    if 0 <= raw_label < num_classes:
        return raw_label

    raise ValueError(
        f"Unexpected phase label value: {raw_label}"
    )


set_seed(SEED)


In [ ]:
# =========================================================
# 3) DATASET
# =========================================================
class SurgicalTurboDataset(Dataset):
    """Load S-STKG fragments through a complete, self-validating cache."""

    REQUIRED_SUFFIXES = (
        "_nodes.txt",
        "_edges.txt",
        "_edge_features.txt",
        "_labels.txt",
    )

    def __init__(
        self,
        root_dir,
        binary_dir,
        chunk_size=100,
        minimum_chunk_frames=10,
    ):
        self.root_dir = Path(root_dir).expanduser().resolve()
        self.binary_dir = Path(binary_dir).expanduser().resolve()
        self.chunk_size = int(chunk_size)
        self.minimum_chunk_frames = int(minimum_chunk_frames)

        if paths_overlap(self.root_dir, self.binary_dir):
            raise RuntimeError(
                "Refusing to use a binary cache that overlaps the source "
                f"data.\nSource: {self.root_dir}\nCache : {self.binary_dir}"
            )

        super().__init__(str(self.root_dir))

        self.case_directories = discover_case_directories(self.root_dir)
        if len(self.case_directories) != EXPECTED_NUM_CASES:
            raise RuntimeError(
                f"Expected {EXPECTED_NUM_CASES} source cases but found "
                f"{len(self.case_directories)}."
            )

        self.source_fingerprint = self._compute_source_fingerprint()
        cache_is_valid, reason = self._cache_is_valid()

        if cache_is_valid:
            print(f"[OK] Complete binary cache is reused: {self.binary_dir}")
        else:
            print(f"[INFO] Binary cache will be rebuilt: {reason}")
            self._convert_and_fragment()

        self.files = sorted(self.binary_dir.glob("frag_*.pt"))
        if not self.files:
            raise RuntimeError(f"Binary cache is empty: {self.binary_dir}")

    @property
    def manifest_path(self):
        return self.binary_dir / "cache_manifest.json"

    @property
    def expected_case_ids(self):
        return [path.name for path in self.case_directories]

    def _source_files(self):
        source_files = []
        for case_directory in self.case_directories:
            for source_file in case_directory.iterdir():
                if (
                    source_file.is_file()
                    and source_file.name.endswith(self.REQUIRED_SUFFIXES)
                ):
                    source_files.append(source_file)
        return sorted(source_files)

    def _compute_source_fingerprint(self):
        source_files = self._source_files()
        if not source_files:
            raise RuntimeError(
                f"No S-STKG text files were found under {self.root_dir}."
            )

        hasher = hashlib.sha256()
        for source_file in source_files:
            stat = source_file.stat()
            relative_path = source_file.relative_to(self.root_dir)
            record = (
                f"{relative_path.as_posix()}|{stat.st_size}|"
                f"{stat.st_mtime_ns}\n"
            )
            hasher.update(record.encode("utf-8"))
        return hasher.hexdigest()

    def _expected_manifest_fields(self):
        return {
            "cache_schema_version": CACHE_SCHEMA_VERSION,
            "source_root": str(self.root_dir),
            "source_fingerprint": self.source_fingerprint,
            "case_ids": self.expected_case_ids,
            "chunk_size": self.chunk_size,
            "minimum_chunk_frames": self.minimum_chunk_frames,
            "frame_key": ["g_ratio", "p_ratio"],
            "feature_cleaning": {
                "case_id_zeroed": bool(REMOVE_CASE_ID_FEATURE),
                "g_ratio_zeroed": bool(REMOVE_G_RATIO_FEATURE),
                "p_ratio_zeroed": bool(REMOVE_P_RATIO_FEATURE),
            },
        }

    def _cache_is_valid(self):
        if not self.binary_dir.is_dir():
            return False, "cache directory does not exist"
        if not self.manifest_path.is_file():
            return False, "cache completion manifest is missing"

        try:
            manifest = json.loads(
                self.manifest_path.read_text(encoding="utf-8")
            )
        except Exception as error:
            return False, f"cache manifest cannot be read: {error}"

        for key, expected_value in self._expected_manifest_fields().items():
            if manifest.get(key) != expected_value:
                return False, (
                    f"manifest field {key!r} does not match the current "
                    "input or configuration"
                )

        fragment_files = sorted(self.binary_dir.glob("frag_*.pt"))
        expected_fragments = int(manifest.get("num_fragments", -1))

        if expected_fragments <= 0:
            return False, "manifest reports no completed fragments"
        if len(fragment_files) != expected_fragments:
            return False, (
                "cache is incomplete: manifest reports "
                f"{expected_fragments} fragments but found "
                f"{len(fragment_files)}"
            )

        return True, "cache is complete"

    @staticmethod
    def _safe_remove_directory(directory, source_root):
        directory = Path(directory).expanduser().resolve()
        source_root = Path(source_root).expanduser().resolve()

        if (
            directory in {Path('/').resolve(), Path.home().resolve()}
            or paths_overlap(directory, source_root)
        ):
            raise RuntimeError(
                f"Unsafe directory-removal request rejected: {directory}"
            )

        if directory.exists():
            shutil.rmtree(directory)

    @staticmethod
    def _pair_mask(table, g_ratio, p_ratio):
        g_mask = np.isclose(
            table.iloc[:, 0].to_numpy(dtype=float),
            float(g_ratio),
            rtol=0.0,
            atol=FRAME_KEY_TOLERANCE,
        )
        p_mask = np.isclose(
            table.iloc[:, 1].to_numpy(dtype=float),
            float(p_ratio),
            rtol=0.0,
            atol=FRAME_KEY_TOLERANCE,
        )
        return g_mask & p_mask

    @staticmethod
    def _validate_tables(n_df, e_df, ef_df, label_df, fragment_prefix):
        if n_df.shape[1] < 11:
            raise RuntimeError(
                f"{fragment_prefix}: node table has {n_df.shape[1]} "
                "columns; expected at least 11."
            )
        if e_df.shape[1] < 5:
            raise RuntimeError(
                f"{fragment_prefix}: edge table has {e_df.shape[1]} "
                "columns; expected at least 5."
            )
        if ef_df.shape[1] < 7:
            raise RuntimeError(
                f"{fragment_prefix}: edge-feature table has "
                f"{ef_df.shape[1]} columns; expected at least 7."
            )
        if label_df.empty or label_df.shape[1] < 2:
            raise RuntimeError(
                f"{fragment_prefix}: label table is empty or invalid."
            )

    def _convert_and_fragment(self):
        print("\n[INFO] Converting role-normalized S-STKGs into binary fragments...")

        temporary_dir = (
            self.binary_dir.parent / f".{self.binary_dir.name}.building"
        )
        self._safe_remove_directory(temporary_dir, self.root_dir)
        temporary_dir.mkdir(parents=True, exist_ok=False)

        global_idx = 0
        case_fragment_counts = {
            case_id: 0 for case_id in self.expected_case_ids
        }

        try:
            for case_directory in tqdm(
                self.case_directories,
                desc="Building source binary cache",
            ):
                prefixes = sorted(
                    {
                        path.name.removesuffix("_nodes.txt")
                        for path in case_directory.glob("*_nodes.txt")
                    }
                )

                if not prefixes:
                    raise RuntimeError(
                        f"No *_nodes.txt files found in {case_directory}."
                    )

                for pref in prefixes:
                    paths = {
                        "nodes": case_directory / f"{pref}_nodes.txt",
                        "edges": case_directory / f"{pref}_edges.txt",
                        "edge_features": (
                            case_directory / f"{pref}_edge_features.txt"
                        ),
                        "labels": case_directory / f"{pref}_labels.txt",
                    }

                    missing = [
                        str(path) for path in paths.values() if not path.is_file()
                    ]
                    if missing:
                        raise FileNotFoundError(
                            f"Incomplete S-STKG fragment {pref}; missing:\n"
                            + "\n".join(missing)
                        )

                    n_df = pd.read_csv(paths["nodes"])
                    e_df = pd.read_csv(paths["edges"])
                    ef_df = pd.read_csv(paths["edge_features"])
                    label_df = pd.read_csv(paths["labels"])

                    self._validate_tables(n_df, e_df, ef_df, label_df, pref)

                    raw_label = int(label_df.iloc[0, 1])
                    label = safe_label_to_zero_based(raw_label, NUM_PHASES)

                    frame_keys = np.unique(
                        n_df.iloc[:, [G_RATIO_COL, P_RATIO_COL]].to_numpy(
                            dtype=float
                        ),
                        axis=0,
                    )
                    if frame_keys.size == 0:
                        continue

                    frame_keys = frame_keys[
                        np.lexsort((frame_keys[:, 1], frame_keys[:, 0]))
                    ]

                    for chunk_start in range(0, len(frame_keys), self.chunk_size):
                        chunk_frame_keys = frame_keys[
                            chunk_start:chunk_start + self.chunk_size
                        ]

                        if len(chunk_frame_keys) < self.minimum_chunk_frames:
                            continue

                        graph_seq = []

                        for frame_g_ratio, frame_p_ratio in chunk_frame_keys:
                            node_mask = (
                                np.isclose(
                                    n_df.iloc[:, G_RATIO_COL].to_numpy(dtype=float),
                                    float(frame_g_ratio),
                                    rtol=0.0,
                                    atol=FRAME_KEY_TOLERANCE,
                                )
                                & np.isclose(
                                    n_df.iloc[:, P_RATIO_COL].to_numpy(dtype=float),
                                    float(frame_p_ratio),
                                    rtol=0.0,
                                    atol=FRAME_KEY_TOLERANCE,
                                )
                            )
                            f_n = n_df.loc[node_mask].to_numpy(
                                dtype=np.float32,
                                copy=True,
                            )

                            if len(f_n) == 0:
                                raise RuntimeError(
                                    f"Empty frame in {pref}: "
                                    f"g_ratio={frame_g_ratio}, "
                                    f"p_ratio={frame_p_ratio}"
                                )

                            if REMOVE_CASE_ID_FEATURE:
                                f_n[:, CASE_ID_COL] = 0.0
                            if REMOVE_G_RATIO_FEATURE:
                                f_n[:, G_RATIO_COL] = 0.0
                            if REMOVE_P_RATIO_FEATURE:
                                f_n[:, P_RATIO_COL] = 0.0

                            edge_mask = self._pair_mask(
                                e_df, frame_g_ratio, frame_p_ratio
                            )
                            edge_feature_mask = self._pair_mask(
                                ef_df, frame_g_ratio, frame_p_ratio
                            )
                            f_e = e_df.loc[edge_mask].to_numpy(copy=True)
                            f_ef = ef_df.loc[edge_feature_mask].to_numpy(copy=True)

                            if len(f_e) != len(f_ef):
                                raise RuntimeError(
                                    "Edge and edge-feature row counts differ for "
                                    f"{pref}, g_ratio={frame_g_ratio}, "
                                    f"p_ratio={frame_p_ratio}: "
                                    f"{len(f_e)} versus {len(f_ef)}"
                                )

                            if len(f_e) > 0:
                                e_idx = torch.tensor(
                                    f_e[:, 2:4].astype(np.int64),
                                    dtype=torch.long,
                                ).t().contiguous()

                                if (
                                    int(e_idx.min()) < 0
                                    or int(e_idx.max()) >= len(f_n)
                                ):
                                    raise RuntimeError(
                                        f"Frame-local edge index is invalid in {pref}."
                                    )
                            else:
                                e_idx = torch.zeros((2, 0), dtype=torch.long)

                            if len(f_ef) > 0:
                                e_at = torch.tensor(
                                    f_ef[:, 4:7].astype(np.float32),
                                    dtype=torch.float,
                                )
                            else:
                                e_at = torch.zeros((0, 3), dtype=torch.float)

                            data = Data(
                                x=torch.tensor(f_n, dtype=torch.float),
                                edge_index=e_idx,
                                edge_attr=e_at,
                                y=torch.tensor([label], dtype=torch.long),
                            )
                            data.is_v = data.x[:, VIRTUAL_NODE_COL] == 1

                            virtual_count = int(data.is_v.sum().item())
                            if virtual_count != 1:
                                raise RuntimeError(
                                    "Expected exactly one virtual node per sampled "
                                    f"frame in {pref}; found {virtual_count} at "
                                    f"g_ratio={frame_g_ratio}, "
                                    f"p_ratio={frame_p_ratio}."
                                )

                            graph_seq.append(data)

                        cache_object = {
                            "graph_seq": graph_seq,
                            "metadata": {
                                "case_id": case_directory.name,
                                "prefix": pref,
                                "chunk_start_gr": float(chunk_frame_keys[0][0]),
                                "chunk_end_gr": float(chunk_frame_keys[-1][0]),
                                "chunk_start_pr": float(chunk_frame_keys[0][1]),
                                "chunk_end_pr": float(chunk_frame_keys[-1][1]),
                                "num_sampled_frames": int(len(graph_seq)),
                                "phase_label_id": int(label),
                                "raw_phase_label": int(raw_label),
                                "feature_cleaning": {
                                    "case_id_zeroed": bool(REMOVE_CASE_ID_FEATURE),
                                    "g_ratio_zeroed": bool(REMOVE_G_RATIO_FEATURE),
                                    "p_ratio_zeroed": bool(REMOVE_P_RATIO_FEATURE),
                                },
                            },
                        }

                        torch.save(
                            cache_object,
                            temporary_dir / f"frag_{global_idx:06d}.pt",
                        )
                        case_fragment_counts[case_directory.name] += 1
                        global_idx += 1

            empty_cases = [
                case_id
                for case_id, fragment_count in case_fragment_counts.items()
                if fragment_count == 0
            ]
            if empty_cases:
                raise RuntimeError(
                    "No retained binary fragment was created for:\n"
                    + "\n".join(empty_cases)
                )
            if global_idx == 0:
                raise RuntimeError("No binary fragments were created.")

            manifest = {
                **self._expected_manifest_fields(),
                "num_fragments": int(global_idx),
                "case_fragment_counts": case_fragment_counts,
                "completed_at": datetime.now().isoformat(),
            }
            (temporary_dir / "cache_manifest.json").write_text(
                json.dumps(manifest, indent=2, ensure_ascii=False),
                encoding="utf-8",
            )

            if self.binary_dir.exists():
                self._safe_remove_directory(self.binary_dir, self.root_dir)
            temporary_dir.rename(self.binary_dir)

        except Exception:
            if temporary_dir.exists():
                self._safe_remove_directory(temporary_dir, self.root_dir)
            raise

        print(f"[OK] Complete binary cache created: {global_idx} fragments.")

    def len(self):
        return len(self.files)

    def _load_raw(self, idx):
        return torch.load(
            self.files[int(idx)],
            map_location="cpu",
            weights_only=False,
        )

    def get_with_meta(self, idx):
        obj = self._load_raw(idx)

        if not isinstance(obj, dict) or "graph_seq" not in obj:
            raise RuntimeError(
                f"Unsupported cache object in {self.files[int(idx)]}."
            )

        graph_seq = obj["graph_seq"]
        meta = dict(obj.get("metadata", {}) or {})
        meta["fragment_file"] = self.files[int(idx)].name

        if not graph_seq:
            raise RuntimeError(
                f"Empty graph sequence in {self.files[int(idx)]}."
            )

        return graph_seq, meta

    def get(self, idx):
        seq, _ = self.get_with_meta(idx)
        return seq

In [ ]:
dataset = SurgicalTurboDataset(
    root_dir=BASE_PATH,
    binary_dir=BIN_PATH,
    chunk_size=CHUNK_SIZE,
    minimum_chunk_frames=MIN_CHUNK_FRAMES,
)

print(f"Binary fragments: {len(dataset)}")


In [ ]:
# =========================================================
# 3) PHASE-ARTIFACT AND SPLIT VALIDATION
# =========================================================

def collect_fragment_metadata(dataset):
    rows = []

    for idx in range(len(dataset)):
        sequence, metadata = dataset.get_with_meta(
            int(idx)
        )
        phase_id = int(
            sequence[0].y.item()
        )

        rows.append(
            {
                "idx": int(idx),
                "case_id": str(
                    metadata.get(
                        "case_id",
                        "UNKNOWN",
                    )
                ),
                "phase_id": phase_id,
                "phase_name": (
                    PHASE_NAMES[phase_id]
                ),
                "fragment_file": str(
                    metadata.get(
                        "fragment_file",
                        "",
                    )
                ),
                "prefix": str(
                    metadata.get(
                        "prefix",
                        "",
                    )
                ),
            }
        )

    metadata_table = pd.DataFrame(rows)

    if metadata_table.empty:
        raise RuntimeError(
            "Current cache contains no fragment metadata."
        )

    return metadata_table


def normalize_metadata_for_comparison(metadata_table):
    required_columns = [
        "idx",
        "case_id",
        "phase_id",
        "fragment_file",
        "prefix",
    ]
    missing_columns = [
        column
        for column in required_columns
        if column not in metadata_table.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Fragment metadata is missing required columns: "
            f"{missing_columns}"
        )

    normalized = (
        metadata_table[
            required_columns
        ]
        .copy()
        .sort_values("idx")
        .reset_index(drop=True)
    )

    normalized["idx"] = normalized[
        "idx"
    ].astype(int)
    normalized["phase_id"] = normalized[
        "phase_id"
    ].astype(int)

    for column in (
        "case_id",
        "fragment_file",
        "prefix",
    ):
        normalized[column] = normalized[
            column
        ].fillna("").astype(str)

    return normalized


def validate_phase_cache_order(
    dataset,
    phase_metadata_csv,
):
    current_metadata = (
        collect_fragment_metadata(dataset)
    )
    phase_metadata = pd.read_csv(
        phase_metadata_csv
    )

    current_normalized = (
        normalize_metadata_for_comparison(
            current_metadata
        )
    )
    phase_normalized = (
        normalize_metadata_for_comparison(
            phase_metadata
        )
    )

    if len(current_normalized) != len(
        phase_normalized
    ):
        raise RuntimeError(
            "Current cache length does not match Notebook 08 metadata: "
            f"{len(current_normalized)} versus "
            f"{len(phase_normalized)}."
        )

    if not current_normalized.equals(
        phase_normalized
    ):
        difference_mask = (
            current_normalized
            != phase_normalized
        ).any(axis=1)
        first_difference = int(
            np.flatnonzero(
                difference_mask.to_numpy()
            )[0]
        )

        raise RuntimeError(
            "Current binary-cache ordering is incompatible with "
            "Notebook 08's saved split. The first mismatch is:\n\n"
            f"Current: "
            f"{current_normalized.iloc[first_difference].to_dict()}\n"
            f"Phase  : "
            f"{phase_normalized.iloc[first_difference].to_dict()}\n\n"
            "Use the same source S-STKG files and the finalized Notebook "
            "08 cache procedure before training the motif extractor."
        )

    print(
        "[OK] Current cache ordering exactly matches "
        "Notebook 08 fragment metadata."
    )

    return current_metadata


def load_saved_split(
    split_npz,
    metadata_table,
):
    split_npz = Path(
        split_npz
    ).expanduser().resolve()

    with np.load(split_npz) as split_data:
        required_keys = {
            "train_idx",
            "val_idx",
            "test_idx",
        }
        missing_keys = (
            required_keys
            - set(split_data.files)
        )

        if missing_keys:
            raise RuntimeError(
                f"Saved split is missing arrays: "
                f"{sorted(missing_keys)}"
            )

        train_idx = np.asarray(
            split_data["train_idx"],
            dtype=int,
        ).reshape(-1)
        val_idx = np.asarray(
            split_data["val_idx"],
            dtype=int,
        ).reshape(-1)
        test_idx = np.asarray(
            split_data["test_idx"],
            dtype=int,
        ).reshape(-1)

    split_arrays = {
        "train": train_idx,
        "val": val_idx,
        "test": test_idx,
    }

    for split_name, indices in (
        split_arrays.items()
    ):
        if len(indices) == 0:
            raise RuntimeError(
                f"Saved {split_name} split is empty."
            )

        if len(np.unique(indices)) != len(
            indices
        ):
            raise RuntimeError(
                f"Duplicate fragment indices were found in "
                f"the {split_name} split."
            )

        if (
            indices.min() < 0
            or indices.max() >= len(metadata_table)
        ):
            raise RuntimeError(
                f"Saved {split_name} indices are incompatible "
                f"with dataset length {len(metadata_table)}."
            )

    train_set = set(train_idx.tolist())
    val_set = set(val_idx.tolist())
    test_set = set(test_idx.tolist())

    if (
        train_set & val_set
        or train_set & test_set
        or val_set & test_set
    ):
        raise RuntimeError(
            "Overlap was detected among train, validation, "
            "and test fragment indices."
        )

    assigned_indices = (
        train_set | val_set | test_set
    )
    expected_indices = set(
        range(len(metadata_table))
    )

    if assigned_indices != expected_indices:
        missing = sorted(
            expected_indices - assigned_indices
        )
        extra = sorted(
            assigned_indices - expected_indices
        )
        raise RuntimeError(
            "Saved split does not partition the complete cache. "
            f"Missing={missing[:10]}, extra={extra[:10]}"
        )

    split_cases = {}
    for split_name, indices in (
        split_arrays.items()
    ):
        split_cases[split_name] = set(
            metadata_table[
                metadata_table["idx"].isin(
                    indices.tolist()
                )
            ]["case_id"].astype(str)
        )

    if (
        split_cases["train"]
        & split_cases["val"]
        or split_cases["train"]
        & split_cases["test"]
        or split_cases["val"]
        & split_cases["test"]
    ):
        raise RuntimeError(
            "Case overlap was detected among the saved splits."
        )

    observed_case_counts = {
        split_name: len(case_ids)
        for split_name, case_ids
        in split_cases.items()
    }
    expected_case_counts = {
        "train": TRAIN_CASES_N,
        "val": VAL_CASES_N,
        "test": TEST_CASES_N,
    }

    if observed_case_counts != expected_case_counts:
        raise RuntimeError(
            "Saved split case counts do not match the required "
            f"14/3/3 protocol: {observed_case_counts}"
        )

    print(
        "[OK] Saved split is complete, case-disjoint, "
        "and compatible with the current cache."
    )

    return train_idx, val_idx, test_idx


def summarize_split(
    dataset,
    train_idx,
    val_idx,
    test_idx,
    save_dir,
):
    save_dir = Path(
        save_dir
    ).expanduser().resolve()
    rows = []

    for split_name, indices in (
        ("train", train_idx),
        ("val", val_idx),
        ("test", test_idx),
    ):
        for idx in indices:
            sequence, metadata = (
                dataset.get_with_meta(
                    int(idx)
                )
            )
            phase_id = int(
                sequence[0].y.item()
            )

            rows.append(
                {
                    "split": split_name,
                    "idx": int(idx),
                    "case_id": str(
                        metadata.get(
                            "case_id",
                            "UNKNOWN",
                        )
                    ),
                    "phase_id": phase_id,
                    "phase_name": (
                        PHASE_NAMES[phase_id]
                    ),
                    "fragment_file": str(
                        metadata.get(
                            "fragment_file",
                            "",
                        )
                    ),
                    "prefix": str(
                        metadata.get(
                            "prefix",
                            "",
                        )
                    ),
                }
            )

    split_table = pd.DataFrame(rows)
    split_table.to_csv(
        save_dir
        / "motif_split_fragment_metadata.csv",
        index=False,
    )

    case_summary = (
        split_table.groupby(
            ["split", "case_id"],
            as_index=False,
        )
        .agg(
            num_fragments=(
                "idx",
                "count",
            )
        )
    )
    case_summary.to_csv(
        save_dir
        / "motif_split_case_summary.csv",
        index=False,
    )

    phase_counts = (
        split_table.groupby(
            ["split", "phase_name"],
            as_index=False,
        )
        .agg(
            num_fragments=(
                "idx",
                "count",
            )
        )
    )
    phase_counts.to_csv(
        save_dir
        / "motif_split_phase_counts.csv",
        index=False,
    )

    print("\n[SPLIT CASE SUMMARY]")
    print(
        case_summary.to_string(
            index=False
        )
    )

    print("\n[SPLIT PHASE COUNTS]")
    print(
        phase_counts.pivot(
            index="phase_name",
            columns="split",
            values="num_fragments",
        )
        .fillna(0)
        .astype(int)
    )


In [ ]:
# =========================================================
# 4) MOTIF-WINDOW CLASS WEIGHTS
# =========================================================

def compute_motif_class_weights(
    dataset,
    indices,
    motif_window=16,
    motif_stride=8,
    num_classes=8,
):
    counts = np.zeros(
        num_classes,
        dtype=np.float64,
    )

    for idx in indices:
        sequence = dataset.get(
            int(idx)
        )
        label = int(
            sequence[0].y.item()
        )
        sequence_length = len(sequence)

        if sequence_length < motif_window:
            num_motifs = 1
        else:
            num_motifs = len(
                range(
                    0,
                    sequence_length
                    - motif_window
                    + 1,
                    motif_stride,
                )
            )

        counts[label] += num_motifs

    counts = np.clip(
        counts,
        a_min=1,
        a_max=None,
    )
    weights = 1.0 / counts
    weights = (
        weights
        / weights.sum()
        * num_classes
    )

    print(
        "\n[MOTIF WINDOW COUNTS]",
        counts.astype(int).tolist(),
    )
    print(
        "[MOTIF CLASS WEIGHTS]",
        [
            round(float(weight), 4)
            for weight in weights.tolist()
        ],
    )

    return torch.tensor(
        weights,
        dtype=torch.float,
    )


In [ ]:
# =========================================================
# 5) MODEL
# =========================================================
class RobustMosaicGNN(nn.Module):
    def __init__(self, in_channels=11, edge_dim=3, hidden_dim=128, num_phases=8, motif_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv1 = GATv2Conv(in_channels, hidden_dim, edge_dim=edge_dim)
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, edge_dim=edge_dim)
        self.context_attn = nn.Linear(hidden_dim * 2, 1)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 3, num_phases)
        self.motif_proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, motif_dim))
        self.motif_cls = nn.Linear(motif_dim, num_phases)
        self.window_fuse = nn.Linear(hidden_dim * 2, hidden_dim)
        self.motif_window = 16
        self.motif_stride = 8

    def encode_frames(self, graph_seq):
        frame_embs = []
        for data in graph_seq:
            x = self.conv1(data.x, data.edge_index, data.edge_attr).relu()
            x = self.conv2(x, data.edge_index, data.edge_attr).relu()
            mean_emb = x.mean(dim=0, keepdim=True)
            v_node_emb = x[data.is_v].mean(dim=0, keepdim=True) if data.is_v.any() else mean_emb
            attn_weight = torch.sigmoid(self.context_attn(torch.cat([mean_emb, v_node_emb], dim=-1)))
            frame_embs.append(attn_weight * v_node_emb + (1.0 - attn_weight) * mean_emb)
        return torch.stack(frame_embs).squeeze(1)

    def apply_temporal_modeling(self, frame_tensor):
        rnn_out, h_n = self.rnn(frame_tensor.unsqueeze(0))
        rnn_out = rnn_out.squeeze(0)
        temporal_out = 0.5 * (rnn_out[:, :self.hidden_dim] + rnn_out[:, self.hidden_dim:])
        temporal_frames = 0.5 * frame_tensor + 0.5 * temporal_out
        rnn_final = torch.cat([h_n[0], h_n[1]], dim=-1)
        global_skip = frame_tensor.mean(dim=0, keepdim=True)
        seq_feature = torch.cat([rnn_final, global_skip], dim=-1)
        return temporal_frames, seq_feature

    def pool_window(self, window_tensor):
        return self.window_fuse(torch.cat([window_tensor.mean(dim=0, keepdim=True), window_tensor[-1].unsqueeze(0)], dim=-1))

    def extract_motifs(self, frame_tensor):
        T = frame_tensor.shape[0]
        if T < self.motif_window:
            return F.normalize(self.motif_proj(self.pool_window(frame_tensor)), dim=-1)
        windows = [self.pool_window(frame_tensor[s:s + self.motif_window]) for s in range(0, T - self.motif_window + 1, self.motif_stride)]
        return F.normalize(self.motif_proj(torch.cat(windows, dim=0)), dim=-1)

    def forward(self, graph_seq):
        frames = self.encode_frames(graph_seq)
        temporal_frames, seq_feature = self.apply_temporal_modeling(frames)
        sigs = self.extract_motifs(temporal_frames)
        return {"motif_logits": self.motif_cls(sigs), "motif_sigs": sigs, "seq_logits": self.fc(seq_feature)}

In [ ]:
# =========================================================
# 6) LOSSES / OPTIMIZER
# =========================================================
def motif_auxiliary_loss(sigs, threshold=0.7):
    if sigs.size(0) <= 1:
        return torch.tensor(0.0, device=sigs.device)
    pull = (1.0 - F.cosine_similarity(sigs[:-1], sigs[1:], dim=-1)).mean()
    sim_mat = sigs @ sigs.t()
    mask = ~torch.eye(sim_mat.size(0), device=sigs.device, dtype=torch.bool)
    off_diag = sim_mat[mask]
    push = torch.relu(off_diag - threshold).mean() if off_diag.numel() > 0 else torch.tensor(0.0, device=sigs.device)
    return pull + 0.5 * push


def compute_total_motif_loss(out, motif_target, seq_target, criterion, aux_weight=0.02, seq_weight=0.20):
    motif_cls_loss = criterion(out["motif_logits"], motif_target)
    motif_aux_loss = motif_auxiliary_loss(out["motif_sigs"])
    seq_cls_loss = criterion(out["seq_logits"], seq_target)
    total_loss = motif_cls_loss + aux_weight * motif_aux_loss + seq_weight * seq_cls_loss
    return total_loss, motif_cls_loss, motif_aux_loss, seq_cls_loss


def apply_partial_unfreeze(model):
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if ("motif_" in name or "conv2" in name or "context_attn" in name or "rnn" in name or "window_fuse" in name or "fc" in name):
            p.requires_grad = True
    print("\n[TRAINABLE PARAMS]")
    for name, p in model.named_parameters():
        if p.requires_grad:
            print("  -", name)


def build_optimizer(model):
    motif_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "motif_" in name or "window_fuse" in name or "fc" in name:
            motif_params.append(p)
        else:
            backbone_params.append(p)
    return torch.optim.AdamW([
        {"params": motif_params, "lr": MOTIF_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ], weight_decay=WEIGHT_DECAY)

In [ ]:
# =========================================================
# 7) EVALUATION AND FIGURE HELPERS
# =========================================================

def compute_basic_metrics(y_true, y_pred):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )
    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    if len(y_true) == 0:
        raise RuntimeError(
            "Cannot compute metrics for an empty prediction set."
        )

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )
    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy),
        "macro_precision": float(
            macro_precision
        ),
        "macro_recall": float(
            macro_recall
        ),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(
            weighted_precision
        ),
        "weighted_recall": float(
            weighted_recall
        ),
        "weighted_f1": float(
            weighted_f1
        ),
    }


def evaluate_split(
    model,
    dataset,
    indices,
    device,
    criterion,
    phase_names,
    aux_weight=0.02,
    seq_weight=0.20,
):
    model.eval()

    total_losses = []
    motif_losses = []
    auxiliary_losses = []
    sequence_losses = []
    y_true = []
    y_pred = []

    with torch.no_grad():
        for idx in indices:
            sequence = [
                graph.to(device)
                for graph in dataset.get(
                    int(idx)
                )
            ]
            output = model(sequence)

            motif_target = (
                sequence[0].y.to(device)
                .repeat(
                    output[
                        "motif_logits"
                    ].size(0)
                )
            )
            sequence_target = (
                sequence[0].y.to(device)
            )

            (
                total_loss,
                motif_loss,
                auxiliary_loss,
                sequence_loss,
            ) = compute_total_motif_loss(
                output,
                motif_target,
                sequence_target,
                criterion,
                aux_weight,
                seq_weight,
            )

            total_losses.append(
                float(total_loss.item())
            )
            motif_losses.append(
                float(motif_loss.item())
            )
            auxiliary_losses.append(
                float(auxiliary_loss.item())
            )
            sequence_losses.append(
                float(sequence_loss.item())
            )

            predictions = (
                output["motif_logits"]
                .argmax(dim=1)
                .cpu()
                .numpy()
            )
            y_pred.extend(
                predictions.tolist()
            )
            y_true.extend(
                [int(sequence[0].y.item())]
                * len(predictions)
            )

    y_true = np.asarray(
        y_true,
        dtype=int,
    )
    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    metrics = compute_basic_metrics(
        y_true,
        y_pred,
    )
    metrics.update(
        {
            "loss_total": float(
                np.mean(total_losses)
            ),
            "loss_motif_cls": float(
                np.mean(motif_losses)
            ),
            "loss_aux": float(
                np.mean(auxiliary_losses)
            ),
            "loss_seq_cls": float(
                np.mean(sequence_losses)
            ),
            "y_true": y_true,
            "y_pred": y_pred,
            "report": classification_report(
                y_true,
                y_pred,
                labels=list(
                    range(len(phase_names))
                ),
                target_names=phase_names,
                zero_division=0,
            ),
        }
    )

    return metrics


def safe_silhouette(
    signatures,
    labels,
    metric="cosine",
    max_samples=4000,
    random_state=42,
):
    labels = np.asarray(labels)

    if signatures is None:
        return np.nan

    num_samples = len(signatures)
    num_labels = len(np.unique(labels))

    if (
        num_samples < 2
        or num_labels < 2
        or num_labels >= num_samples
    ):
        return np.nan

    sample_size = (
        max_samples
        if len(signatures) > max_samples
        else None
    )

    return float(
        silhouette_score(
            signatures,
            labels,
            metric=metric,
            sample_size=sample_size,
            random_state=random_state,
        )
    )


def analyze_motif_representation(
    model,
    dataset,
    indices,
    device,
):
    model.eval()
    all_signatures = []
    all_labels = []

    with torch.no_grad():
        for idx in indices:
            sequence = [
                graph.to(device)
                for graph in dataset.get(
                    int(idx)
                )
            ]
            output = model(sequence)

            signatures = (
                output["motif_sigs"]
                .cpu()
                .numpy()
            )
            all_signatures.append(
                signatures
            )
            all_labels.extend(
                [int(sequence[0].y.item())]
                * len(signatures)
            )

    if not all_signatures:
        return np.nan, None, None

    all_signatures = np.concatenate(
        all_signatures,
        axis=0,
    )
    all_labels = np.asarray(
        all_labels,
        dtype=int,
    )

    phase_silhouette = safe_silhouette(
        all_signatures,
        all_labels,
        metric="cosine",
    )

    return (
        phase_silhouette,
        all_signatures,
        all_labels,
    )


def analyze_unsupervised_structure(
    signatures,
    n_clusters=8,
    random_state=42,
):
    if (
        signatures is None
        or signatures.shape[0] <= n_clusters
        or n_clusters < 2
    ):
        return np.nan, None, None

    kmeans = KMeans(
        n_clusters=n_clusters,
        n_init=20,
        random_state=random_state,
    )
    cluster_ids = kmeans.fit_predict(
        signatures
    )
    silhouette = safe_silhouette(
        signatures,
        cluster_ids,
        metric="cosine",
    )
    cluster_sizes = np.bincount(
        cluster_ids,
        minlength=n_clusters,
    )

    return (
        silhouette,
        cluster_ids,
        cluster_sizes,
    )


def plot_confusion_matrix(
    matrix,
    save_path,
    title,
):
    figure, axis = plt.subplots(
        figsize=(10, 8)
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=[
            f"P{phase_id + 1}"
            for phase_id in range(
                NUM_PHASES
            )
        ],
    )
    display.plot(
        ax=axis,
        values_format="d",
        colorbar=False,
    )
    axis.set_title(title)
    figure.tight_layout()
    figure.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(figure)


def plot_history(
    history,
    save_path,
):
    epochs = np.arange(
        1,
        len(history["train_loss"]) + 1,
    )

    figure, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5),
    )

    axes[0].plot(
        epochs,
        history["train_loss"],
        marker="o",
        label="Train total loss",
    )
    axes[0].plot(
        epochs,
        history["val_loss"],
        marker="s",
        label="Validation total loss",
    )
    axes[0].set_title(
        "Total Loss"
    )
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(
        True,
        alpha=0.3,
    )

    axes[1].plot(
        epochs,
        history["train_acc"],
        marker="o",
        label="Train accuracy",
    )
    axes[1].plot(
        epochs,
        history["val_acc"],
        marker="s",
        label="Validation accuracy",
    )
    axes[1].set_title(
        "Accuracy"
    )
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(
        True,
        alpha=0.3,
    )

    figure.tight_layout()
    figure.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(figure)


def finite_or_none(value):
    value = float(value)

    if np.isfinite(value):
        return value

    return None


In [ ]:
# =========================================================
# 8) COMPLETE MOTIF-EXTRACTOR PIPELINE
# =========================================================

def validate_phase_checkpoint(
    checkpoint,
):
    if "model_state_dict" not in checkpoint:
        raise RuntimeError(
            "The selected phase checkpoint does not contain "
            "'model_state_dict'."
        )

    if checkpoint.get(
        "role_normalized",
        True,
    ) is not True:
        raise RuntimeError(
            "The selected checkpoint is not marked as "
            "role-normalized."
        )

    checkpoint_phase_names = checkpoint.get(
        "phase_names"
    )
    if (
        checkpoint_phase_names is not None
        and list(checkpoint_phase_names)
        != PHASE_NAMES
    ):
        raise RuntimeError(
            "Phase-name ordering in the checkpoint does not "
            "match this notebook."
        )

    expected_cleaning = {
        "case_id_zeroed": bool(
            REMOVE_CASE_ID_FEATURE
        ),
        "g_ratio_zeroed": bool(
            REMOVE_G_RATIO_FEATURE
        ),
        "p_ratio_zeroed": bool(
            REMOVE_P_RATIO_FEATURE
        ),
    }
    checkpoint_cleaning = checkpoint.get(
        "feature_cleaning"
    )

    if (
        checkpoint_cleaning is not None
        and checkpoint_cleaning
        != expected_cleaning
    ):
        raise RuntimeError(
            "Feature-cleaning settings do not match the "
            "selected phase checkpoint.\n"
            f"Expected: {expected_cleaning}\n"
            f"Found   : {checkpoint_cleaning}"
        )

    checkpoint_config = checkpoint.get(
        "config",
        {},
    )
    if (
        "chunk_size" in checkpoint_config
        and int(
            checkpoint_config["chunk_size"]
        ) != CHUNK_SIZE
    ):
        raise RuntimeError(
            "CHUNK_SIZE does not match the selected phase "
            "checkpoint."
        )

    print(
        "[OK] Phase-checkpoint metadata is compatible."
    )


def load_phase_weights(
    model,
    checkpoint,
):
    load_result = model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=False,
    )

    expected_missing_prefixes = (
        "motif_proj.",
        "motif_cls.",
        "window_fuse.",
    )
    unexpected_missing = [
        key
        for key in load_result.missing_keys
        if not key.startswith(
            expected_missing_prefixes
        )
    ]

    if unexpected_missing:
        raise RuntimeError(
            "Unexpected phase-initialization parameters are "
            f"missing: {unexpected_missing}"
        )

    if load_result.unexpected_keys:
        raise RuntimeError(
            "Unexpected keys were found in the phase "
            f"checkpoint: {load_result.unexpected_keys}"
        )

    print("\n[PHASE CHECKPOINT LOAD]")
    print(
        "Motif-specific parameters initialized from scratch:",
        load_result.missing_keys,
    )
    print(
        "Unexpected checkpoint keys:",
        load_result.unexpected_keys,
    )


def run_motif_pipeline():
    set_seed(SEED)
    validate_input_paths()

    results_root = Path(
        RESULTS_ROOT
    ).expanduser().resolve()
    results_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )
    results_dir = (
        results_root
        / (
            "phase_initialized_motif_extractor_"
            f"seed{SEED}_{timestamp}"
        )
    )
    results_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    model_save_path = (
        results_dir
        / "best_phase_initialized_motif_extractor.pt"
    )

    print("\n" + "=" * 90)
    print(
        "PHASE-INITIALIZED S-STKG MOTIF EXTRACTOR"
    )
    print("=" * 90)
    print(f"Output directory : {results_dir}")
    print(f"Source S-STKG    : {BASE_PATH}")
    print(f"Binary cache     : {BIN_PATH}")
    print(f"Phase checkpoint : {PHASE_CKPT}")
    print(f"Saved split      : {SPLIT_NPZ}")
    print(f"Device           : {DEVICE}")

    dataset = SurgicalTurboDataset(
        root_dir=BASE_PATH,
        binary_dir=BIN_PATH,
        chunk_size=CHUNK_SIZE,
        minimum_chunk_frames=MIN_CHUNK_FRAMES,
    )

    current_metadata = (
        validate_phase_cache_order(
            dataset,
            PHASE_METADATA_CSV,
        )
    )
    (
        train_idx,
        val_idx,
        test_idx,
    ) = load_saved_split(
        SPLIT_NPZ,
        current_metadata,
    )

    summarize_split(
        dataset,
        train_idx,
        val_idx,
        test_idx,
        results_dir,
    )

    shutil.copy2(
        SPLIT_NPZ,
        results_dir / "data_split.npz",
    )

    model = RobustMosaicGNN(
        hidden_dim=HIDDEN_DIM,
        num_phases=NUM_PHASES,
        motif_dim=MOTIF_DIM,
    ).to(DEVICE)

    phase_checkpoint = torch.load(
        PHASE_CKPT,
        map_location=DEVICE,
        weights_only=False,
    )
    validate_phase_checkpoint(
        phase_checkpoint
    )
    load_phase_weights(
        model,
        phase_checkpoint,
    )

    apply_partial_unfreeze(model)

    class_weights = (
        compute_motif_class_weights(
            dataset,
            train_idx,
            model.motif_window,
            model.motif_stride,
            NUM_PHASES,
        )
        .to(DEVICE)
    )

    optimizer = build_optimizer(model)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=LABEL_SMOOTHING,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_macro_f1": [],
        "val_macro_f1": [],
    }

    best_val_macro_f1 = float("-inf")
    best_val_acc = float("-inf")
    best_epoch = -1

    print("\n[TRAINING STARTED]")

    for epoch in range(
        1,
        NUM_EPOCHS + 1,
    ):
        model.train()
        total_loss_sum = 0.0
        y_true_train = []
        y_pred_train = []

        progress = tqdm(
            train_idx,
            desc=f"Epoch {epoch:02d}",
        )

        for idx in progress:
            sequence = [
                graph.to(DEVICE)
                for graph in dataset.get(
                    int(idx)
                )
            ]

            optimizer.zero_grad(
                set_to_none=True
            )
            output = model(sequence)

            motif_target = (
                sequence[0].y.to(DEVICE)
                .repeat(
                    output[
                        "motif_logits"
                    ].size(0)
                )
            )
            sequence_target = (
                sequence[0].y.to(DEVICE)
            )

            (
                total_loss,
                motif_classification_loss,
                motif_signature_loss,
                sequence_classification_loss,
            ) = compute_total_motif_loss(
                output,
                motif_target,
                sequence_target,
                criterion,
                AUX_WEIGHT,
                SEQ_WEIGHT,
            )

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRAD_CLIP_NORM,
            )
            optimizer.step()

            total_loss_sum += float(
                total_loss.item()
            )

            predictions = (
                output["motif_logits"]
                .argmax(dim=1)
                .detach()
                .cpu()
                .numpy()
            )
            y_pred_train.extend(
                predictions.tolist()
            )
            y_true_train.extend(
                [int(sequence[0].y.item())]
                * len(predictions)
            )

            progress.set_postfix(
                total=f"{total_loss.item():.4f}",
                motif=(
                    f"{motif_classification_loss.item():.4f}"
                ),
                signature=(
                    f"{motif_signature_loss.item():.4f}"
                ),
                sequence=(
                    f"{sequence_classification_loss.item():.4f}"
                ),
            )

        train_metrics = compute_basic_metrics(
            y_true_train,
            y_pred_train,
        )
        train_loss = (
            total_loss_sum
            / max(len(train_idx), 1)
        )

        val_metrics = evaluate_split(
            model,
            dataset,
            val_idx,
            DEVICE,
            criterion,
            PHASE_NAMES,
            AUX_WEIGHT,
            SEQ_WEIGHT,
        )

        history["train_loss"].append(
            train_loss
        )
        history["val_loss"].append(
            val_metrics["loss_total"]
        )
        history["train_acc"].append(
            train_metrics["accuracy"]
            * 100.0
        )
        history["val_acc"].append(
            val_metrics["accuracy"]
            * 100.0
        )
        history["train_macro_f1"].append(
            train_metrics["macro_f1"]
        )
        history["val_macro_f1"].append(
            val_metrics["macro_f1"]
        )

        print(
            f"\nEpoch {epoch:02d} | "
            f"Train loss {train_loss:.4f} | "
            f"Val loss {val_metrics['loss_total']:.4f} | "
            f"Train acc "
            f"{train_metrics['accuracy'] * 100.0:.2f}% | "
            f"Val acc "
            f"{val_metrics['accuracy'] * 100.0:.2f}% | "
            f"Train Macro F1 "
            f"{train_metrics['macro_f1']:.4f} | "
            f"Val Macro F1 "
            f"{val_metrics['macro_f1']:.4f}"
        )

        validation_macro_f1 = float(
            val_metrics["macro_f1"]
        )
        validation_accuracy = float(
            val_metrics["accuracy"]
            * 100.0
        )

        save_checkpoint = (
            validation_macro_f1
            > best_val_macro_f1
            or (
                abs(
                    validation_macro_f1
                    - best_val_macro_f1
                )
                < 1e-8
                and validation_accuracy
                > best_val_acc
            )
        )

        if save_checkpoint:
            best_val_macro_f1 = (
                validation_macro_f1
            )
            best_val_acc = (
                validation_accuracy
            )
            best_epoch = int(epoch)

            torch.save(
                {
                    "epoch": int(epoch),
                    "model_state_dict": (
                        model.state_dict()
                    ),
                    "val_acc": float(
                        best_val_acc
                    ),
                    "val_macro_f1": float(
                        best_val_macro_f1
                    ),
                    "role_normalized": True,
                    "phase_checkpoint": str(
                        Path(PHASE_CKPT)
                        .expanduser()
                        .resolve()
                    ),
                    "split_npz": str(
                        results_dir
                        / "data_split.npz"
                    ),
                    "base_path": str(
                        Path(BASE_PATH)
                        .expanduser()
                        .resolve()
                    ),
                    "binary_cache": str(
                        Path(BIN_PATH)
                        .expanduser()
                        .resolve()
                    ),
                    "phase_names": PHASE_NAMES,
                    "feature_cleaning": {
                        "case_id_zeroed": bool(
                            REMOVE_CASE_ID_FEATURE
                        ),
                        "g_ratio_zeroed": bool(
                            REMOVE_G_RATIO_FEATURE
                        ),
                        "p_ratio_zeroed": bool(
                            REMOVE_P_RATIO_FEATURE
                        ),
                    },
                    "config": {
                        "seed": int(SEED),
                        "chunk_size": int(
                            CHUNK_SIZE
                        ),
                        "minimum_chunk_frames": int(
                            MIN_CHUNK_FRAMES
                        ),
                        "hidden_dim": int(
                            HIDDEN_DIM
                        ),
                        "motif_dim": int(
                            MOTIF_DIM
                        ),
                        "motif_window": int(
                            model.motif_window
                        ),
                        "motif_stride": int(
                            model.motif_stride
                        ),
                        "num_epochs": int(
                            NUM_EPOCHS
                        ),
                        "aux_weight": float(
                            AUX_WEIGHT
                        ),
                        "seq_weight": float(
                            SEQ_WEIGHT
                        ),
                        "label_smoothing": float(
                            LABEL_SMOOTHING
                        ),
                        "motif_lr": float(
                            MOTIF_LR
                        ),
                        "backbone_lr": float(
                            BACKBONE_LR
                        ),
                        "weight_decay": float(
                            WEIGHT_DECAY
                        ),
                        "gradient_clip_norm": float(
                            GRAD_CLIP_NORM
                        ),
                    },
                },
                model_save_path,
            )

            print(
                "  [SAVE] New best motif checkpoint | "
                f"Val Macro F1={best_val_macro_f1:.4f} | "
                f"Val Acc={best_val_acc:.2f}%"
            )

    if not model_save_path.is_file():
        raise RuntimeError(
            "Training completed without creating a motif "
            "checkpoint."
        )

    best_checkpoint = torch.load(
        model_save_path,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )

    train_evaluation = evaluate_split(
        model,
        dataset,
        train_idx,
        DEVICE,
        criterion,
        PHASE_NAMES,
        AUX_WEIGHT,
        SEQ_WEIGHT,
    )
    validation_evaluation = evaluate_split(
        model,
        dataset,
        val_idx,
        DEVICE,
        criterion,
        PHASE_NAMES,
        AUX_WEIGHT,
        SEQ_WEIGHT,
    )
    test_evaluation = evaluate_split(
        model,
        dataset,
        test_idx,
        DEVICE,
        criterion,
        PHASE_NAMES,
        AUX_WEIGHT,
        SEQ_WEIGHT,
    )

    (
        phase_silhouette,
        signatures,
        signature_labels,
    ) = analyze_motif_representation(
        model,
        dataset,
        test_idx,
        DEVICE,
    )

    (
        unsupervised_silhouette,
        cluster_ids,
        cluster_sizes,
    ) = analyze_unsupervised_structure(
        signatures,
        NUM_PHASES,
        SEED,
    )

    test_confusion_matrix = (
        confusion_matrix(
            test_evaluation["y_true"],
            test_evaluation["y_pred"],
            labels=list(
                range(NUM_PHASES)
            ),
        )
    )

    confusion_matrix_path = (
        results_dir
        / "motif_extractor_confusion_matrix.png"
    )
    plot_confusion_matrix(
        test_confusion_matrix,
        confusion_matrix_path,
        (
            "Phase-Initialized Motif Extractor\n"
            f"Accuracy: "
            f"{test_evaluation['accuracy'] * 100.0:.2f}% | "
            f"Macro F1: "
            f"{test_evaluation['macro_f1']:.4f}"
        ),
    )

    history_path = (
        results_dir
        / "motif_extractor_training_history.png"
    )
    plot_history(
        history,
        history_path,
    )

    history_table = pd.DataFrame(
        {
            "epoch": np.arange(
                1,
                len(history["train_loss"]) + 1,
            ),
            **history,
        }
    )
    history_table.to_csv(
        results_dir
        / "motif_extractor_training_history.csv",
        index=False,
    )

    report_path = (
        results_dir
        / "phase_initialized_motif_extractor_report.txt"
    )

    with report_path.open(
        "w",
        encoding="utf-8",
    ) as report_file:
        report_file.write(
            "=" * 90 + "\n"
        )
        report_file.write(
            "PHASE-INITIALIZED S-STKG MOTIF "
            "EXTRACTOR REPORT\n"
        )
        report_file.write(
            "=" * 90 + "\n\n"
        )
        report_file.write(
            f"Phase checkpoint: {PHASE_CKPT}\n"
        )
        report_file.write(
            f"Copied split: "
            f"{results_dir / 'data_split.npz'}\n"
        )
        report_file.write(
            f"Best epoch: {best_epoch}\n"
        )
        report_file.write(
            "Best validation Macro F1: "
            f"{best_val_macro_f1:.4f}\n"
        )
        report_file.write(
            "Best validation accuracy: "
            f"{best_val_acc:.2f}%\n"
        )
        report_file.write(
            "Test accuracy: "
            f"{test_evaluation['accuracy'] * 100.0:.2f}%\n"
        )
        report_file.write(
            "Test Macro F1: "
            f"{test_evaluation['macro_f1']:.4f}\n"
        )
        report_file.write(
            "Test weighted F1: "
            f"{test_evaluation['weighted_f1']:.4f}\n"
        )
        report_file.write(
            "Phase-label cosine silhouette (test): "
            f"{phase_silhouette}\n"
        )
        report_file.write(
            "Unsupervised KMeans cosine silhouette (test): "
            f"{unsupervised_silhouette}\n"
        )

        if cluster_sizes is not None:
            report_file.write(
                "Unsupervised cluster sizes: "
                f"{cluster_sizes.tolist()}\n"
            )

        for split_name, evaluation in (
            ("TRAIN", train_evaluation),
            (
                "VALIDATION",
                validation_evaluation,
            ),
            ("TEST", test_evaluation),
        ):
            report_file.write(
                "\n" + "-" * 90 + "\n"
            )
            report_file.write(
                f"{split_name} METRICS\n"
            )
            report_file.write(
                "-" * 90 + "\n"
            )
            report_file.write(
                "Total loss: "
                f"{evaluation['loss_total']:.4f}\n"
            )
            report_file.write(
                "Motif classification loss: "
                f"{evaluation['loss_motif_cls']:.4f}\n"
            )
            report_file.write(
                "Signature loss: "
                f"{evaluation['loss_aux']:.4f}\n"
            )
            report_file.write(
                "Sequence classification loss: "
                f"{evaluation['loss_seq_cls']:.4f}\n"
            )
            report_file.write(
                "Accuracy: "
                f"{evaluation['accuracy'] * 100.0:.2f}%\n"
            )
            report_file.write(
                "Macro F1: "
                f"{evaluation['macro_f1']:.4f}\n"
            )
            report_file.write(
                "Weighted F1: "
                f"{evaluation['weighted_f1']:.4f}\n\n"
            )
            report_file.write(
                evaluation["report"] + "\n"
            )

    run_config = {
        "experiment": (
            "phase_initialized_sstkg_motif_extractor"
        ),
        "base_path": str(
            Path(BASE_PATH)
            .expanduser()
            .resolve()
        ),
        "binary_cache": str(
            Path(BIN_PATH)
            .expanduser()
            .resolve()
        ),
        "phase_results_dir": str(
            Path(PHASE_RESULTS_DIR)
            .expanduser()
            .resolve()
        ),
        "phase_checkpoint": str(
            Path(PHASE_CKPT)
            .expanduser()
            .resolve()
        ),
        "split_npz": str(
            results_dir
            / "data_split.npz"
        ),
        "results_dir": str(
            results_dir
        ),
        "best_model_path": str(
            model_save_path
        ),
        "best_epoch": int(
            best_epoch
        ),
        "best_val_macro_f1": float(
            best_val_macro_f1
        ),
        "best_val_acc": float(
            best_val_acc
        ),
        "test_accuracy": float(
            test_evaluation["accuracy"]
            * 100.0
        ),
        "test_macro_f1": float(
            test_evaluation["macro_f1"]
        ),
        "test_weighted_f1": float(
            test_evaluation["weighted_f1"]
        ),
        "phase_label_cosine_silhouette_test": (
            finite_or_none(
                phase_silhouette
            )
        ),
        "unsupervised_kmeans_cosine_silhouette_test": (
            finite_or_none(
                unsupervised_silhouette
            )
        ),
        "cluster_sizes": (
            cluster_sizes.tolist()
            if cluster_sizes is not None
            else None
        ),
        "feature_cleaning": {
            "case_id_zeroed": bool(
                REMOVE_CASE_ID_FEATURE
            ),
            "g_ratio_zeroed": bool(
                REMOVE_G_RATIO_FEATURE
            ),
            "p_ratio_zeroed": bool(
                REMOVE_P_RATIO_FEATURE
            ),
        },
    }

    (
        results_dir
        / "run_config.json"
    ).write_text(
        json.dumps(
            run_config,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("\n" + "=" * 90)
    print("[COMPLETED]")
    print(
        f"Best motif checkpoint: {model_save_path}"
    )
    print(
        f"Copied split         : "
        f"{results_dir / 'data_split.npz'}"
    )
    print(
        f"Report               : {report_path}"
    )
    print(
        f"Test accuracy        : "
        f"{test_evaluation['accuracy'] * 100.0:.2f}%"
    )
    print(
        f"Test Macro F1        : "
        f"{test_evaluation['macro_f1']:.4f}"
    )
    print("=" * 90)

    return {
        "results_directory": results_dir,
        "checkpoint_path": model_save_path,
        "split_path": (
            results_dir / "data_split.npz"
        ),
        "report_path": report_path,
        "history_path": history_path,
        "confusion_matrix_path": (
            confusion_matrix_path
        ),
        "metrics": {
            "test_accuracy": float(
                test_evaluation["accuracy"]
                * 100.0
            ),
            "test_macro_f1": float(
                test_evaluation["macro_f1"]
            ),
            "test_weighted_f1": float(
                test_evaluation[
                    "weighted_f1"
                ]
            ),
        },
    }


In [ ]:
run_output = run_motif_pipeline()

run_output
